In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_J.xpt"
csv_file = DATA_DIR / "OSQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
# IMPORTANT:
#
# OSQ_J requires NO validated numeric corrections.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 OSQ_J XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OSQ010A",
    "OSQ010B",
    "OSQ010C",
    "OSQ020A",
    "OSQ020B",
    "OSQ020C",
    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",
    "OSD030AB",
    "OSQ040AB",
    "OSD050AB",
    "OSD030AC",
    "OSQ040AC",
    "OSD050AC",
    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",
    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",
    "OSD030BC",
    "OSQ040BC",
    "OSD050BC",
    "OSD030BD",
    "OSQ040BD",
    "OSD050BD",
    "OSD030BE",
    "OSQ040BE",
    "OSD050BE",
    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",
    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",
    "OSD030CC",
    "OSQ040CC",
    "OSD050CC",
    "OSD030CD",
    "OSQ040CD",
    "OSD050CD",
    "OSD030CE",
    "OSQ040CE",
    "OSD050CE",
    "OSQ080",
    "OSQ090A",
    "OSQ100A",
    "OSD110A",
    "OSQ120A",
    "OSQ090B",
    "OSQ100B",
    "OSD110B",
    "OSQ120B",
    "OSQ090C",
    "OSQ100C",
    "OSD110C",
    "OSQ120C",
    "OSQ090D",
    "OSQ100D",
    "OSD110D",
    "OSQ120D",
    "OSQ090E",
    "OSQ100E",
    "OSD110E",
    "OSQ120E",
    "OSQ090F",
    "OSQ100F",
    "OSD110F",
    "OSQ120F",
    "OSQ090G",
    "OSQ120G",
    "OSQ090H",
    "OSQ120H",
    "OSQ090I",
    "OSQ120I",
    "OSQ090J",
    "OSQ100J",
    "OSD110J",
    "OSQ120J",
    "OSQ060",
    "OSQ072",
    "OSQ130",
    "OSQ140Q",
    "OSQ140U",
    "OSQ150",
    "OSQ160A",
    "OSQ160B",
    "OSQ170",
    "OSQ180",
    "OSQ190",
    "OSQ200",
    "OSQ210",
    "OSQ220"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OSQ_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3069:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 95:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,069 rows x 95 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Attached OSQ_J was inspected BEFORE conversion.
#
# No genuine fractional values occur in any of the
# 95 released fields.
#
# Therefore all variables in THIS file are whole-number:
#
#     respondent IDs
#     questionnaire codes
#     fracture counts
#     ages
#     duration quantities
#     categorical codes
#
# IMPORTANT:
#
# This applies ONLY to OSQ_J.
#
# Do NOT generalize this to Dietary, Laboratory,
# Examination, survey weights, nutrients, exposure
# measurements, or other genuine decimal variables.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT/pandas artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value unless the exact
# variable/count has first been validated against CDC.
#
# Direct inspection of attached OSQ_J:
#
#     Expected tiny artifacts = 0
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 95 OSQ_J fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-VALUE CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in OSQ_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: OSQ_J contains no tiny XPORT artifacts."
)


# ============================================================
# 6. CHECK ORIGINAL ORDINARY NUMERIC ZEROS
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL NUMERIC ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zero values "
        "found in OSQ_J."
    )


print(
    "PASS: Original OSQ_J contains no numeric zeros."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# OSQ_J requires ZERO numeric repairs.
#
# Therefore:
#
#     expected_df == original source_df
#
# numerically.
#

expected_df = source_df.copy()


# ============================================================
# 8. CHECK ALL 95 FIELDS FOR GENUINE FRACTIONS
# ============================================================

#
# CRITICAL:
#
# Run this BEFORE converting anything to Int64.
#
# np.round() below is ONLY used as a comparison.
#
# It does NOT modify the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in OSQ_J.\n\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(fractional_columns)
    )


print(
    "\nPASS: No genuine fractional values "
    "in any of the 95 OSQ_J fields."
)


# ============================================================
# 9. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102952:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102952 validated."
)


# ============================================================
# 10. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every OSQ_J field was checked above.
#
# NO:
#
#     df.round()
#
# is used for transformation.
#
# Int64 only changes representation:
#
#     93705.0 -> 93705
#     1.0     -> 1
#     80.0    -> 80
#
# Numeric meaning remains unchanged.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 95 OSQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. CDC EXACT-FREQUENCY HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - OSQ010A
# ============================================================

check_frequency(
    "OSQ010A",
    {
        1: 77,
        2: 2989,
        9: 2
    },
    1
)


# ============================================================
# 13. CDC CHECK - OSQ010B
# ============================================================

check_frequency(
    "OSQ010B",
    {
        1: 315,
        2: 2750,
        9: 3
    },
    1
)


# ============================================================
# 14. CDC CHECK - OSQ010C
# ============================================================

check_frequency(
    "OSQ010C",
    {
        1: 116,
        2: 2948,
        9: 4
    },
    1
)


# ============================================================
# 15. CDC CHECK - OSQ020A
# ============================================================

check_frequency(
    "OSQ020A",
    {
        1: 68,
        2: 7,
        3: 2
    },
    2992
)


# ============================================================
# 16. CDC CHECK - OSQ020B
# ============================================================

check_frequency(
    "OSQ020B",
    {
        1: 252,
        2: 46,
        3: 10,
        4: 3,
        5: 2,
        9999: 2
    },
    2754
)


# ============================================================
# 17. CDC CHECK - OSQ020C
# ============================================================

check_frequency(
    "OSQ020C",
    {
        1: 94,
        2: 15,
        3: 4,
        4: 1,
        5: 1,
        9999: 1
    },
    2953
)


# ============================================================
# 18. CDC CHECK - OSQ080
# ============================================================

check_frequency(
    "OSQ080",
    {
        1: 763,
        2: 2294,
        9: 11
    },
    1
)


# ============================================================
# 19. CDC CHECK - OSQ060
# ============================================================

check_frequency(
    "OSQ060",
    {
        1: 396,
        2: 2657,
        7: 1,
        9: 14
    },
    1
)


# ============================================================
# 20. CDC CHECK - OSQ072
# ============================================================

check_frequency(
    "OSQ072",
    {
        1: 210,
        2: 182,
        9: 4
    },
    2673
)


# ============================================================
# 21. CDC CHECK - OSQ130
# ============================================================

check_frequency(
    "OSQ130",
    {
        1: 238,
        2: 2801,
        9: 29
    },
    1
)


# ============================================================
# 22. CDC CHECK - OSQ140U
# ============================================================

check_frequency(
    "OSQ140U",
    {
        1: 146,
        2: 84
    },
    2839
)


# ============================================================
# 23. CDC CHECK - OSQ150
# ============================================================

check_frequency(
    "OSQ150",
    {
        1: 380,
        2: 2492,
        7: 3,
        9: 193
    },
    1
)


# ============================================================
# 24. CDC CHECK - OSQ160A / OSQ160B
# ============================================================

check_frequency(
    "OSQ160A",
    {
        1: 358
    },
    2711
)


check_frequency(
    "OSQ160B",
    {
        2: 32
    },
    3037
)


# ============================================================
# 25. CDC CHECK - OSQ170
# ============================================================

check_frequency(
    "OSQ170",
    {
        1: 253,
        2: 2728,
        9: 87
    },
    1
)


# ============================================================
# 26. CDC CHECK - OSQ190
# ============================================================

check_frequency(
    "OSQ190",
    {
        1: 2,
        2: 5,
        9: 1
    },
    3061
)


# ============================================================
# 27. CDC CHECK - OSQ200
# ============================================================

check_frequency(
    "OSQ200",
    {
        1: 85,
        2: 2794,
        7: 1,
        9: 188
    },
    1
)


# ============================================================
# 28. CDC CHECK - OSQ220
# ============================================================

check_frequency(
    "OSQ220",
    {
        1: 6,
        2: 2
    },
    3061
)


# ============================================================
# 29. RANGE-SUMMARY HELPER
# ============================================================

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- CDC RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        f"{range_count:,}"
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs from CDC.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            actual_count
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} CDC range summary matches."
    )


# ============================================================
# 30. CDC RANGE CHECK - OSQ140Q
# ============================================================

check_range_summary(
    "OSQ140Q",
    1,
    42,
    230,
    {
        999: 8
    },
    2831
)


# ============================================================
# 31. CDC RANGE CHECK - OSQ180
# ============================================================

check_range_summary(
    "OSQ180",
    7,
    103,
    245,
    {
        999: 8
    },
    2816
)


# ============================================================
# 32. CDC RANGE CHECK - OSQ210
# ============================================================

check_range_summary(
    "OSQ210",
    4,
    99,
    77,
    {
        999: 8
    },
    2984
)


# ============================================================
# 33. ROUTING HELPER
# ============================================================

def check_route(
    description,
    expected_mask,
    child_column
):

    expected = (
        expected_mask
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    present = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )

    mismatch_count = int(
        np.sum(
            expected != present
        )
    )

    print(
        f"\n--- ROUTING: {description} ---"
    )

    print(
        "Expected populated:",
        int(expected.sum())
    )

    print(
        "Actually populated:",
        int(present.sum())
    )

    print(
        "Routing mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"Routing validation failed: "
            f"{description}"
        )

    print(
        "PASS: Routing validated."
    )


# ============================================================
# 34. FRACTURE QUESTION ROUTING
# ============================================================

check_route(
    "OSQ010A=1 -> OSQ020A",
    df["OSQ010A"].eq(1),
    "OSQ020A"
)


check_route(
    "OSQ010B=1 -> OSQ020B",
    df["OSQ010B"].eq(1),
    "OSQ020B"
)


check_route(
    "OSQ010C=1 -> OSQ020C",
    df["OSQ010C"].eq(1),
    "OSQ020C"
)


# ============================================================
# 35. OTHER-FRACTURE ROUTING
# ============================================================

check_route(
    "OSQ080=1 -> OSQ090A",
    df["OSQ080"].eq(1),
    "OSQ090A"
)


# ============================================================
# 36. OSTEOPOROSIS MEDICATION ROUTING
# ============================================================

check_route(
    "OSQ060=1 -> OSQ072",
    df["OSQ060"].eq(1),
    "OSQ072"
)


# ============================================================
# 37. PREDNISONE / CORTISONE ROUTING
# ============================================================

check_route(
    "OSQ130=1 -> OSQ140Q",
    df["OSQ130"].eq(1),
    "OSQ140Q"
)


#
# OSQ140U is populated only when OSQ140Q contains
# an ordinary duration quantity 1-42.
#
check_route(
    "OSQ140Q 1-42 -> OSQ140U",
    df["OSQ140Q"].between(
        1,
        42,
        inclusive="both"
    ),
    "OSQ140U"
)


# ============================================================
# 38. PARENT OSTEOPOROSIS ROUTING
# ============================================================

#
# OSQ160A and OSQ160B are CODE-ALL-THAT-APPLY fields.
#
# Therefore OSQ150=1 should correspond to at least
# one of OSQ160A / OSQ160B being populated.
#

parent_detail_present = (
    df["OSQ160A"].notna()
    |
    df["OSQ160B"].notna()
)


parent_yes = (
    df["OSQ150"]
    .eq(1)
    .fillna(False)
)


parent_route_mismatches = int(
    np.sum(
        parent_yes.to_numpy(
            dtype=bool
        )
        !=
        parent_detail_present.to_numpy(
            dtype=bool
        )
    )
)


print(
    "\n--- OSQ150 / OSQ160 ROUTING ---"
)

print(
    "OSQ150 = Yes:",
    int(parent_yes.sum())
)

print(
    "At least one OSQ160 field populated:",
    int(parent_detail_present.sum())
)

print(
    "Routing mismatches:",
    parent_route_mismatches
)


if parent_route_mismatches != 0:

    raise ValueError(
        "OSQ150 parent osteoporosis routing failed."
    )


print(
    "PASS: All 380 OSQ150=Yes records "
    "have parent detail information."
)


# ============================================================
# 39. MOTHER HIP-FRACTURE ROUTING
# ============================================================

check_route(
    "OSQ170=1 -> OSQ180",
    df["OSQ170"].eq(1),
    "OSQ180"
)


#
# When exact maternal age is unknown (999),
# OSQ190 asks whether she was under/over 50.
#

check_route(
    "OSQ180=999 -> OSQ190",
    df["OSQ180"].eq(999),
    "OSQ190"
)


# ============================================================
# 40. FATHER HIP-FRACTURE ROUTING
# ============================================================

check_route(
    "OSQ200=1 -> OSQ210",
    df["OSQ200"].eq(1),
    "OSQ210"
)


check_route(
    "OSQ210=999 -> OSQ220",
    df["OSQ210"].eq(999),
    "OSQ220"
)


# ============================================================
# 41. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

#
# Original source:
#
#     tiny artifacts = 0
#     ordinary zeros = 0
#
# No numeric repairs were performed.
#
# Therefore final zero pattern must remain empty.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zero values introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zero values introduced."
)


# ============================================================
# 42. COMPLETE VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Permitted numeric modifications in OSQ_J:
#
#     NONE
#
# Therefore all 95 current variables must exactly
# equal their original XPT-derived values.
#

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 95 OSQ_J variables exactly match "
    "their original XPT numeric values."
)


# ============================================================
# 43. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "before CSV export."
    )


print(
    "PASS: Missing positions unchanged."
)


# ============================================================
# 44. EXPORT CSV
# ============================================================

#
# IMPORTANT FIDELITY POLICY
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     blanket decimal formatting
#
# OSQ_J itself contains no genuine decimal fields.
#
# However, this same policy protects genuine decimals
# in your Dietary / Lab / Exam / exposure files.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 45. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 46. EXACT ORIGINAL XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT ORIGINAL XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 95 CSV variables round-trip "
    "to exact original XPT numeric values."
)


# ============================================================
# 47. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 48. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    3069,
    95
):

    raise ValueError(
        "Raw-text CSV dimensions changed."
    )


# ============================================================
# 49. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts "
        "found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 50. RAW CSV .0 CHECK
# ============================================================

#
# Every OSQ_J field was proven to be integral.
#
# Therefore it is safe to check all 95 fields for
# unnecessary ".0".
#
# IMPORTANT:
#
# Do NOT apply this all-column check to files that
# contain genuine decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any of the 95 OSQ_J fields."
)


# ============================================================
# 51. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 52. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 53. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102952:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93705-102952 valid "
    "and contains no '.0'."
)


# ============================================================
# 54. FINAL COMPLETE SOURCE -> CSV FIDELITY
# ============================================================

#
# One final comparison across ALL 95 released fields.
#

final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    original_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    same = np.allclose(
        original_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves exact original "
    "numeric values for all 95 OSQ_J variables."
)


# ============================================================
# 55. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "OSQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - Osteoporosis"
)

print(
    "Eligible population: Age 50+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Original ordinary numeric zeros:",
    true_zero_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "OSQ010A hip fracture = Yes:",
    int(
        df["OSQ010A"]
        .eq(1)
        .sum()
    )
)

print(
    "OSQ010B wrist fracture = Yes:",
    int(
        df["OSQ010B"]
        .eq(1)
        .sum()
    )
)

print(
    "OSQ010C spine fracture = Yes:",
    int(
        df["OSQ010C"]
        .eq(1)
        .sum()
    )
)

print(
    "OSQ060 osteoporosis diagnosis = Yes:",
    int(
        df["OSQ060"]
        .eq(1)
        .sum()
    )
)

print(
    "OSQ072 osteoporosis medication = Yes:",
    int(
        df["OSQ072"]
        .eq(1)
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 OSQ_J XPT -> CSV
Rows: 3,069
Columns: 95
PASS: Structure = 3,069 rows x 95 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: 95

--- ORIGINAL XPT TINY-VALUE CHECK ---
Tiny XPORT artifacts: 0
PASS: OSQ_J contains no tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
Ordinary numeric zeros: 0
PASS: Original OSQ_J contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
OSQ010A: 0 fractional observations
OSQ010B: 0 fractional observations
OSQ010C: 0 fractional observations
OSQ020A: 0 fractional observations
OSQ020B: 0 fractional observations
OSQ020C: 0 fractional observations
OSD030AA: 0 fractional observations
OSQ040AA: 0 fractional observations
OSD050AA: 0 fractional observations
OSD030AB: 0 fractional observations
OSQ040AB: 0 fractional observations
OSD050AB: 0 fractional observations
OSD030AC: 0 fractional observations
OSQ040AC: 0 fractional observations
OSD050AC: 0 fractional observa